In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.measure import label, regionprops
from skimage.segmentation import expand_labels, find_boundaries, clear_border
from skimage.morphology import dilation, square
from skimage import graph
from collections import defaultdict
from tqdm import tqdm
import networkx as nx
import pickle


In [ ]:
CLASS_VALUES = {'vessel': 0, 'ray': 64, 'fiber': 128, 'parenchyma': 192, 'stuff': 255}
V_CLASSES = [CLASS_VALUES['vessel']]
AP_CLASSES = [CLASS_VALUES['parenchyma']]
RAG_CLASSES = V_CLASSES + AP_CLASSES + [CLASS_VALUES['ray']]
VALID_CLASSES = RAG_CLASSES + [CLASS_VALUES['fiber']]


gt_dir = 'dataset/performance_check/mask'
img_ids = [
    'acapod2_01_msk.png', 'alllit2_01_msk.png', 'burspi1_01_msk.png',
    'euclon1_01_msk.png', 'exostr8_01_msk.png', 'hakdac2_01_msk.png'
]

records = []

for img_id in tqdm(img_ids):
    img_path = os.path.join(gt_dir, img_id)
    mask = np.array(Image.open(img_path).convert('L'))

    rag_mask = np.isin(mask, VALID_CLASSES).astype(np.uint8)
    
    raw_label = label(rag_mask)
    
    expanded = expand_labels(raw_label, distance=40)

    expanded = clear_border(expanded)
    

    post_class_map = {}
    for region in regionprops(expanded):
        coords = region.coords
        values = mask[coords[:, 0], coords[:, 1]]
        most_common_class = np.bincount(values).argmax()
        post_class_map[region.label] = most_common_class


    

    unique_labels = np.unique(expanded)
    vessel_mask = np.zeros_like(expanded, dtype=bool)
    
    for label_id in unique_labels:
        if post_class_map.get(label_id, 255) == CLASS_VALUES['vessel']:
            vessel_mask[expanded == label_id] = True
    

    vessel_labels_post = label(vessel_mask)
    props = regionprops(vessel_labels_post)
    expanded_vessel_ids = np.unique(vessel_labels_post)[1:]
    print(f"number of vessl labels after expansion: {len(expanded_vessel_ids)}")

    small_vessels = [p for p in props if p.area <= 5]
    print(f"number of vessels of area≦5px: {len(small_vessels)}")

    
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(raw_label, cmap='nipy_spectral')
    axes[0].set_title(f"Before expand: {img_id}")
    axes[1].imshow(expanded, cmap='nipy_spectral')
    axes[1].set_title("After expand_labels")
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


    class_map = {}
    for region in regionprops(expanded):
        coords = region.coords
        values = mask[coords[:, 0], coords[:, 1]]
        most_common_class = np.bincount(values).argmax()
        class_map[region.label] = most_common_class
    

    rag = graph.rag_boundary(expanded, mask.astype(float), connectivity=1)


    for node in rag.nodes:
        class_val = class_map.get(node)
        if class_val == CLASS_VALUES['vessel']:
            rag.nodes[node]['anatomy'] = 'vessel'
        elif class_val == CLASS_VALUES['ray']:
            rag.nodes[node]['anatomy'] = 'ray'
        elif class_val == CLASS_VALUES['parenchyma']:
            rag.nodes[node]['anatomy'] = 'parenchyma'
        elif class_val == CLASS_VALUES['fiber']:
            rag.nodes[node]['anatomy'] = 'fiber'
        else:
            rag.nodes[node]['anatomy'] = 'unknown'
        

    expanded_save_dir = 'expanded_labels'
    os.makedirs(expanded_save_dir, exist_ok=True)
    expanded_filename = img_id.replace('_msk.png', '_expanded.npy')
    expanded_path = os.path.join(expanded_save_dir, expanded_filename)
    np.save(expanded_path, expanded)


    props = regionprops(expanded)
    area_dict = {r.label: r.area for r in props}
    boundaries = find_boundaries(expanded, mode='inner')
    perim_dict = {r.label: np.sum(boundaries[expanded == r.label]) for r in props}

    for n1, n2 in rag.edges:
        c1, c2 = class_map.get(n1), class_map.get(n2)
        if c1 in V_CLASSES and c2 in AP_CLASSES:
            v, ap = n1, n2
        elif c2 in V_CLASSES and c1 in AP_CLASSES:
            v, ap = c2, n1
        else:
            continue

        v_mask = (expanded == v)
        ap_mask = (expanded == ap)
        v_boundary = find_boundaries(v_mask, mode='inner')
        contact_px = np.sum(v_boundary & dilation(ap_mask, square(3)))

  
        
        v_area = area_dict.get(v, 1)
        ap_area = area_dict.get(ap, 1)
        v_perim = perim_dict.get(v, 1)

        records.append({
            'image': img_id,
            'v_id': v,
            'ap_id': ap,
            'contact_px': contact_px,
            'v_perim': v_perim,
            'v_area': v_area,
            'ap_area': ap_area,
            'contact_per_perim': contact_px / v_perim,
            'ap_area_per_v_area': ap_area / v_area
        })


    graph_save_dir = 'graphs_vpr'
    os.makedirs(graph_save_dir, exist_ok=True)

    graph_filename = img_id.replace('_msk.png', '_rag.gpickle')
    graph_path = os.path.join(graph_save_dir, graph_filename)
       
    with open(graph_path, 'wb') as f:
        pickle.dump(rag, f)


df = pd.DataFrame(records)


In [ ]:
#df.to_csv('df_for_weighted_degree.csv', index=False)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.measure import label, regionprops
from skimage.segmentation import expand_labels, find_boundaries
from skimage.morphology import dilation, square
from skimage.io import imread
from skimage import graph
from collections import defaultdict
from tqdm import tqdm
import networkx as nx
import pickle


In [ ]:

df = pd.read_csv('df_for_weighted_degree.csv')

In [ ]:

CLASS_VALUES = {'vessel': 0, 'ray': 64, 'fiber': 128, 'parenchyma': 192, 'stuff': 255}
V_CLASSES = [CLASS_VALUES['vessel']]


gt_dir = 'dataset/performance_check/mask'
expanded_dir = 'expanded_labels'
img_ids = [
    'acapod2_01_msk.png', 'alllit2_01_msk.png', 'burspi1_01_msk.png',
    'euclon1_01_msk.png', 'exostr8_01_msk.png', 'hakdac2_01_msk.png'
]


vessel_props_records = []

for img_id in tqdm(img_ids):

    mask_path = os.path.join(gt_dir, img_id)
    expanded_path = os.path.join(expanded_dir, img_id.replace('_msk.png', '_expanded.npy'))
    
    mask = np.array(Image.open(mask_path).convert('L'))
    expanded = np.load(expanded_path)

    class_map = {}
    for region in regionprops(expanded):
        coords = region.coords
        values = mask[coords[:, 0], coords[:, 1]]
        most_common_class = np.bincount(values).argmax()
        class_map[region.label] = most_common_class


    for region in regionprops(expanded):
        label_id = region.label
        if class_map.get(label_id) == CLASS_VALUES['vessel']:
            vessel_props_records.append({
                'image': img_id,
                'v_id': label_id,
                'v_area': region.area,
                'v_perim': np.sum(find_boundaries(expanded == label_id, mode='inner'))
            })


df_props = pd.DataFrame(vessel_props_records)


In [ ]:

df = df[df['v_area'] > 0.9]

# Vissualisation no.1: Vessel perimeter vs contact length
if not df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    for name, group in df.groupby('image'):
        ax1.scatter(group['v_perim'], group['contact_px'], label=name, alpha=0.7)
    ax1.set_xlabel('Vessel Perimeter (px)')
    ax1.set_ylabel('Contact Length with Parenchyma (px)')
    ax1.set_title('V–AP Contact Length vs Vessel Perimeter')
    ax1.legend()

# Vissualisation no.2: Vessel area vs AP area
    for name, group in df.groupby('image'):
        ax2.scatter(group['v_area'], group['ap_area'], label=name, alpha=0.7)
    ax2.set_xlabel('Vessel Area (px)')
    ax2.set_ylabel('Adjacent Parenchyma Area (px)')
    ax2.set_title('Adjacent AP Area vs Vessel Area')
    ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ V–AP was not detected")


In [ ]:
df_filtered = df[df['v_area'] > 1]

# 必要な列だけに絞って保存
df_filtered[['image', 'v_id', 'ap_id', 'contact_px', 'v_perim', 'v_area', 'ap_area',
             'contact_per_perim', 'ap_area_per_v_area']].to_csv('filtered_edges.csv', index=False)


In [ ]:
print(df_filtered)

## Weighted degree

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


df_filtered['v_radius'] = np.sqrt(df_filtered['v_area'] / np.pi)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))


for name, group in df_filtered.groupby('image'):
    ax1.scatter(group['contact_per_perim'], group['ap_area_per_v_area'], label=name, alpha=0.7)
ax1.set_xlabel('Contact Length / Vessel Perimeter')
ax1.set_ylabel('Parenchyma Area / Vessel Area')
ax1.set_title('V–AP Edge Metrics per Image')
ax1.legend(title='Image ID')
ax1.grid(True)


for name, group in df_filtered.groupby('image'):
    ax2.scatter(
        group['contact_per_perim'],
        group['ap_area_per_v_area'],
        s=group['v_radius']**2 * 0.005,
        label=name,
        alpha=0.6
    )
ax2.set_xlabel('Contact Length / Vessel Perimeter')
ax2.set_ylabel('Parenchyma Area / Vessel Area')
ax2.set_title('Size-Scaled V–AP Metrics')
ax2.legend(title='Image ID')
ax2.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.preprocessing import MinMaxScaler


df_all_vessels = df_props[['image', 'v_id', 'v_perim', 'v_area']].copy()


df_vap_summary = (
    df_filtered
    .groupby(['image', 'v_id'])
    .agg({
        'contact_px': 'sum',
        'ap_area': 'sum'
    })
    .reset_index()
)


df_vessel_summary = pd.merge(df_all_vessels, df_vap_summary, on=['image', 'v_id'], how='left')


df_vessel_summary[['contact_px', 'ap_area']] = df_vessel_summary[['contact_px', 'ap_area']].fillna(0)


df_vessel_summary['contact_per_perim'] = df_vessel_summary['contact_px'] / df_vessel_summary['v_perim']
df_vessel_summary['ap_area_per_v_area'] = df_vessel_summary['ap_area'] / df_vessel_summary['v_area']
df_vessel_summary['v_radius'] = np.sqrt(df_vessel_summary['v_area'] / np.pi)

scaler = MinMaxScaler()
df_vessel_summary[['contact_norm', 'ap_area_norm']] = scaler.fit_transform(
    df_vessel_summary[['contact_per_perim', 'ap_area_per_v_area']]
)

df_vessel_summary['Integrated Weighted Degree'] = (
    0.5 * df_vessel_summary['contact_norm'] +
    0.5 * df_vessel_summary['ap_area_norm']
)


In [ ]:

os.makedirs("plots_dfs", exist_ok=True)

df_vessel_summary.to_csv(
    "plots_dfs/vessel_level_connectivity_summary.csv",
    index=False
)

In [ ]:

plt.figure(figsize=(8, 6))

for name, group in df_vessel_summary.groupby('image'):
    plt.scatter(
        group['contact_per_perim'],
        group['ap_area_per_v_area'],
        s=group['v_radius']**2 * 0.015,
        label=name,
        alpha=0.7
    )
mpl.rcParams['pdf.fonttype'] = 42
plt.xlabel('Total Contact Length / Vessel Perimeter')
plt.ylabel('Total Parenchyma Area / Vessel Area')
plt.title('V–AP Interaction Metrics per Vessel')
plt.legend(title='Image ID')
plt.grid(True)
plt.tight_layout()

save_path = 'plots_dfs/weighted_degree.pdf'

plt.savefig(save_path)

plt.show()
